# Phase 2 - Heart Disease Generative AI Advice

**Course:** SWE485 (Selected Topics in Software Engineering)
**Phase:** 2 (Generative AI)

# The Notebook Overview

### Main Objective



The core objective of this generative AI component is to generate a personalised health overview for each patient based on their heart disease classification result and their individual clinical feature values. Given a patient's prediction label and clinical measurements, the system identifies the key patterns present in that patient's data and produces an interpreted overview of their cardiovascular health profile grounded in those individual values, not in generic disease labels.
The overview is designed to translate the model's classification output into plain language, helping the patient understand what their measurements indicate, which values fall outside the healthy range, and what the overall pattern of their data suggests about their cardiovascular status. It may also include general wellness tips where relevant to the patient's specific measurements. This positions the system as a health literacy tool that bridges the gap between a raw classification result and a meaningful, readable interpretation of that assessment.


This objective is shared identically across all four prompt templates. The input is the same for all templates and the output is always a personalised health overview. The four templates differ in their prompting techniques and target user profiles, enabling a broader evaluation of how these factors influence the quality, clarity, and depth of the generated overview.

#### Prompt Inputs

The system uses two primary inputs for each patient assessment: the prediction label and the clinical feature values. Each input plays a specific and deliberate role in enabling the generative model to produce a personalised and clinically grounded advisory.

**Prediction Label**
The prediction label anchors the entire advisory. It tells the system
whether the patient has been classified as having heart disease or not,
which determines the overall direction and tone of the generated advice.
Without this label, the system would have no starting point for framing
the response.

**Clinical Feature Values: Before One Hot Encoding**
The feature values are passed to the generative model in their original
categorical and numerical form, before any one hot encoding is applied.
This is a deliberate and important design decision.

One hot encoding is a preprocessing technique designed for machine
learning algorithms that require numerical inputs. It transforms a
categorical feature such as ChestPainType into multiple binary columns
such as ChestPainType_ASY, ChestPainType_ATA, and so on. While this
transformation is necessary for the supervised model to process the
data, it produces a representation that is fragmented, sparse, and
entirely unreadable to a human or a language model. Traditional
preprocessing methods like one hot encoding can inflate the feature
space and reduce interpretability, particularly with high-dimensional
categorical variables (Ali et al., 2026). Passing one hot encoded
features to a generative AI model would result in the model receiving
inputs like ChestPainType_ASY = 1 and ChestPainType_ATA = 0, which
carry no natural language meaning and cannot be used to generate
coherent clinical explanations.

By contrast, passing the original feature value such as
ChestPainType = ASY allows the generative model to understand the
meaning of the input directly, reason about it using its clinical
knowledge, and incorporate it naturally into the generated advisory.
Large language models are trained on natural language and structured
clinical text, which means they are far better equipped to interpret
and reason about human-readable feature values than binary encoded
representations (Sivarajkumar et al., 2024). This approach ensures
that the advisory is grounded in clinically meaningful inputs rather
than machine-optimised numerical encodings that obscure the original
meaning of the data.

---

> Ali, A.S., et al. (2026). *Unveiling patterns in clinical data:
> exploring the role of large language models and clustering algorithms.*
> Frontiers in Artificial Intelligence, 9, 1737530.
> https://doi.org/10.3389/frai.2026.1737530

> Sivarajkumar, S., et al. (2024). *An empirical evaluation of prompting
> strategies for large language models in zero-shot clinical natural
> language processing.* JMIR Medical Informatics, 12, e55318.
> https://doi.org/10.2196/55318

#### Handling Output Scenarios

The advisory generated by the system is not binary. It does not simply say
"you are healthy" or "you have heart disease." Instead, the response is
always anchored to the patient's specific clinical numbers, and the logic
of the advisory adapts based on two dimensions: the prediction label and
the individual feature values.

**For patients classified as Heart Disease Detected:**
The system identifies which specific clinical features are contributing to
the risk and generates advice directly tied to those features. Two patients
with the same label but different clinical values will always receive
different advice because the advisory is driven by the individual numbers,
not the label alone.

**For patients classified as No Heart Disease Detected:**
The system does not simply reassure the patient. Instead it evaluates the individual feature values using its embedded clinical knowledge to determine whether they are within healthy or concerning ranges. Two distinct
scenarios are handled:

The first scenario is when the patient's feature values are within healthy
ranges. In this case the advisory acknowledges the healthy status and
encourages the patient to maintain their current lifestyle and habits,
reinforcing positive health behaviours.

The second scenario is when one or more feature values are borderline,
meaning they are technically within the healthy classification range but
are approaching concerning thresholds. In this case the system generates
a targeted warning for that specific feature, advising the patient to
monitor it and take preventive action before it progresses. This makes
the advisory genuinely personalised even for patients who are currently
classified as healthy.

This approach reflects the principle that effective health advisory systems
should go beyond binary classification outputs and provide individuals with
actionable, context-aware guidance tailored to their unique clinical profile
(Li et al., 2024). Research in personalised preventive healthcare confirms
that advice anchored to individual clinical values, rather than population-
level generalisations, produces more meaningful and actionable outcomes for
patients (Bajwa et al., 2021). Furthermore, addressing borderline values
in healthy patients aligns with the broader goal of preventive medicine,
where early identification of risk trajectories is considered more valuable
than waiting for a disease threshold to be crossed (Topol, 2019).

---

> Li, Y.H., Li, Y.L., & Wei, M.Y. (2024). *Innovation and challenges of
> artificial intelligence technology in personalized healthcare.*
> Scientific Reports, 14, 18994.
> https://doi.org/10.1038/s41598-024-70073-7

> Bajwa, J., Munir, U., Nori, A., & Williams, B. (2021). *Artificial
> intelligence in healthcare: Transforming the practice of medicine.*
> Future Healthcare Journal, 8(2), e188–e194.
> https://doi.org/10.7861/fhj.2021-0095

> Topol, E.J. (2019). *High-performance medicine: The convergence of
> human and artificial intelligence.* Nature Medicine, 25(1), 44–56.
> https://doi.org/10.1038/s41591-018-0300-7


### Shared Prompt Design Principles
 **System and User Prompt Separation**

The prompt structure adopted across all four templates in this project is divided into two distinct parts: a System Instruction and a User Prompt. This design follows the Google Gemini API documentation, which provides a dedicated `system_instruction` parameter that separates the model's persistent behavioural rules from the variable user message (Google, 2024). The system instruction defines the model's role, tone, language level, and constraints, which are rules that must remain fixed for that template regardless of which patient is being assessed. The user prompt then carries the patient-specific data and the task-related description for each patient assessment.

This separation is not merely a technical convention. It reflects a fundamental principle in prompt engineering: that governance rules and task-specific content should never be mixed in the same layer, as doing so risks inconsistency, ambiguity, and unpredictable model behaviour (Tetrate, 2025). When behavioural rules are embedded within the user prompt, they become prone to being overridden, forgotten, or misinterpreted. By contrast, placing them in a dedicated system instruction ensures they are treated with higher priority by the model and remain stable across all requests.

This principle is especially critical in a medical context. In healthcare AI applications, consistency is not a preference but a safety requirement. A system that gives different advice to similar patients, or that drops its safety constraints when the user prompt is complex, poses a direct risk to patient trust and clinical validity. As noted by Gharib et al. (2024), effective prompt design in healthcare requires that the model's behavioural boundaries are clearly established and reliably maintained across all interactions. Similarly, research on prompt engineering in clinical practice emphasises that small changes in prompt structure can significantly impact output quality and safety in medical settings (Cascella et al., 2024). By anchoring all safety rules, role definitions, and tone constraints within the system instruction, each template in this project ensures that no matter what patient data is passed through the user prompt, the model will always respond within the same controlled and medically appropriate framework.

> Google. (2024). *Gemini API documentation: System instructions.*
> Google AI for Developers.
> https://ai.google.dev/gemini-api/docs/system-instructions

> Tetrate. (2025). *System prompts vs user prompts: Design patterns
> for LLM apps.*
> https://tetrate.io/learn/ai/system-prompts-vs-user-prompts

> Gharib, A.M., et al. (2024). *A guide to prompt design: Foundations
> and applications for healthcare simulationists.* Frontiers in Medicine,
> 11, 1504532.
> https://doi.org/10.3389/fmed.2024.1504532

> Cascella, M., Montomoli, J., Bellini, V. and Bignami, E. (2024).
> *Prompt Engineering in Healthcare.* Electronics, 13(15), 2961.
> https://doi.org/10.3390/electronics13152961


**Role Definition**

The system instruction begins by assigning the model a specific professional role as a clinical decision-support assistant specialized in cardiovascular health analysis. This role is intentionally designed to align with the nature of the dataset used in the system, which includes structured patient information such as key health indicators (e.g., cholesterol levels, blood pressure, and other risk-related features).
Establishing a clear role for the model is considered a best practice in prompt engineering. For example, Google’s prompt design guidelines recommend assigning a role to guide the model’s tone, vocabulary, and depth of explanation (Google, 2024). By framing the model as a healthcare-oriented assistant, the generated responses become more consistent, medically relevant, and aligned with real-world clinical reasoning. This ensures that explanations of predicted outcomes are not only technically accurate but also meaningful to end users.
Furthermore, the assigned role encourages the model to interpret predictions in a way that reflects clinical awareness and responsibility, focusing on explaining possible causes, highlighting key contributing factors, and suggesting reasonable next steps when appropriate. At the same time, the system maintains a balance between professionalism and accessibility, ensuring that explanations remain understandable to non-expert users such as patients or general audiences.
This role-based instruction is particularly important given that the system relies on a predictive model trained on patient data stored in the database . By grounding the model’s responses in this role, the system ensures that all generated explanations are context-aware, structured, and aligned with the intended use of the application as an AI-powered health support tool.
Overall, defining the model’s role enhances the quality, reliability, and clarity of the generated explanations, while also supporting the system’s goal of delivering personalized and user-friendly insights based on patient data.


> Google. (2024). *Introduction to prompting.*
> Google AI for Developers.
> https://ai.google.dev/gemini-api/docs/prompting-intro

**Safety Constraints (What the Model Must Never Do)**

Safety constraints are a set of explicit negative rules that define what the model must never do, regardless of the prompting technique or patient data provided. In a cardiovascular health advisory system, the absence of explicit safety constraints creates real clinical and ethical risk. Guidelines for human-AI interaction recommend that AI systems deployed in sensitive contexts must avoid overconfidence, remain transparent about their limitations, and use language appropriate to the user's emotional state (Amershi et al., 2019). Without an explicit prohibition, a model may produce diagnostic-sounding language because such language is statistically common in the medical text the model was trained on. The constraint must therefore be stated directly in every template, because the model will not infer it from context alone. That said, while safety constraints appear in every template, their specific form differs depending on the intended use case. 

> Amershi, S., Weld, D., Vorvoreanu, M., Fourney, A., Nushi, B., Collisson, P., Suh, J., Iqbal, S., Bennett, P., Inkpen, K., Teevan, J., Kikin-Gil, R., and Horvitz, E. (2019). *Guidelines for human-AI interaction.* Proceedings of the 2019 CHI Conference on Human Factors in Computing Systems, 1–13.
> https://doi.org/10.1145/3290605.3300233

**Style Prompting (Tone and Language Control)**

Another component present across all templates is the explicit instruction to control tone and language style. This is formally classified as Style Prompting, a technique that modifies how the model expresses its output rather than what it reasons about or how it structures its response (Sahoo et al., 2024). Style prompting does not change the content, it shapes the way that content is delivered, including the level of formality, the emotional register, and the vocabulary choices the model makes.

In a cardiovascular health advisory system, tone control has a direct impact on how the patient receives and acts on the information. Research in human-computer interaction confirms that in health-related contexts, warm and conversational language builds significantly more trust than clinical or detached language, and that patients are more likely to engage with recommendations delivered in a tone that feels supportive rather than authoritative (Bickmore and Picard, 2005). Without explicit style instructions, a model with access to medical training data will default to clinical register precise, impersonal, and often alarming to a non-expert reader because that is the dominant tone in the text it learned from.

However, the specific style instructions differ from one template to another based on the intended use case and user profile. The appropriate tone for one user type would be mismatched and potentially counterproductive for another, which is why style prompting must be calibrated individually for each template rather than applied uniformly across the system.

> Sahoo, P., Singh, A.K., Saha, S., Jain, V., Mondal, S. and Chadha, A. (2024). *A systematic survey of prompt engineering in large language models: Techniques and applications.*
> https://arxiv.org/pdf/2406.06608

> Bickmore, T. and Picard, R. (2005). *Establishing and maintaining long-term human-computer relationships.* ACM Transactions on Computer-Human Interaction, 12(2), 293–327.
> https://dl.acm.org/doi/10.1145/1067860.1067867

### Prompt Template Documentation Structure

Each prompt template in this notebook is documented following a structured, stage-based approach. Rather than simply presenting the prompt and its output, each template is walked through seven progressive stages: Define, Justify, Plan, Build, Test, Evaluate and Reflect. This ensures that every design decision is transparent, deliberate, and well-reasoned. The **Define** stage establishes what the prompting technique is and why it suits the medical domain. The **Justify** stage explains the specific use case and the rationale behind choosing this approach. The **Plan** stage outlines the prompt engineering techniques applied before writing the prompt if applicable. The **Build** stage presents the actual prompt structure. The **Test** stage demonstrates the template with real examples. Finally, the **Evaluate** and **Reflect** stages assess the output quality and acknowledge the template's limitations and ethical considerations. This progressive structure is applied consistently across all four templates to allow meaningful comparison and analysis.

| # | Phase/Stage | Section | What to Write |
|---|-------------|---------|---------------|
| 1 | | **ID** | A number |
| 2 | Define | **Approach Definition** | Define the prompting technique in simple terms. What is it? How does it work? |
| 3 | Define | **Relation to Medical Field** | Why does this specific technique make sense in a healthcare context? Connect the technique's characteristics to medical needs. |
| 4 | Justify | **Intended Use Case** |  Who is the user? What do they input? What do they need as output? |
| 5 | Justify | **Design Rationale** | Why did we choose this approach over other techniques? What advantages does it have for your specific problem? |
| 6 | Plan | **Prompt Engineering Techniques Applied** | Explain techniques applied in the prompt. Why was each technique included? |
| 7 | Build | **Prompt Structure** | The actual prompt we send to the AI model with placeholders in `{}` for parts that change per user. |
| 8 | Test | **Input / Output Example** | Fill the placeholders with real examples and show the actual AI response you received (3 test cases). |
| 9 | Evaluate | **Evaluation Summary** | Rate the response across the decided criteria. |
| 10 | Reflect | **Assumptions & Limitations** | What must be true for this template to work? Where does it fail or produce weak results? It also includes any ethical concerns. |

# Generative AI Model Setup

### Load the Best Model from Phase 1

In [3]:
import pickle

# Load the best model saved from Phase 1 (XGBoost model)
with open("Supervised_Learning/models/best_xgb.pkl", "rb") as f:
    model = pickle.load(f)

In [4]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import StandardScaler
from sklearn.preprocessing import RobustScaler

#Method to get prediction from the model, this will be used in the prompt for Gemini

#Step 1:
# Fit scalers on the same training data statistics
# Age → StandardScaler | RestingBP → RobustScaler | MaxHR → StandardScaler | Oldpeak → RobustScaler
raw_df =  pd.read_csv("Dataset/preprocessed_heart_data.csv")

#WE NEED TO CHECK THIS!!
age_scaler      = StandardScaler().fit(raw_df[['Age']])
resting_bp_scaler = RobustScaler().fit(raw_df[['RestingBP']])
max_hr_scaler   = StandardScaler().fit(raw_df[['MaxHR']])
oldpeak_scaler  = RobustScaler().fit(raw_df[['Oldpeak']])

def get_prediction(inputs):

    #Step 2: Cholesterol → category then drop raw value 
    chol = inputs["Cholesterol"]
    chol_bins   = [0, 200, 240, np.inf]
    chol_labels = ["Desirable", "Borderline High", "High"]
    chol_cat    = pd.cut([chol], bins=chol_bins, labels=chol_labels, right=False)[0]

    # Step 3:Build encoded row 
    encoded = {
        "Age":                              inputs["Age"],
        "Sex":                              1 if inputs["Sex"] == "Male" else 0,
        "RestingBP":                        inputs["RestingBP"],
        "FastingBS":                        1 if inputs["FastingBS"] > 120 else 0,
        "MaxHR":                            inputs["MaxHR"],
        "ExerciseAngina":                   1 if inputs["ExerciseAngina"] == "Yes" else 0,
        "Oldpeak":                          inputs["Oldpeak"],
        # ChestPainType one-hot
        "ChestPainType_ASY":                1 if inputs["ChestPainType"] == "ASY" else 0,
        "ChestPainType_ATA":                1 if inputs["ChestPainType"] == "ATA" else 0,
        "ChestPainType_NAP":                1 if inputs["ChestPainType"] == "NAP" else 0,
        "ChestPainType_TA":                 1 if inputs["ChestPainType"] == "TA"  else 0,

        # RestingECG one-hot
        "RestingECG_LVH":                   1 if inputs["RestingECG"] == "LVH"    else 0,
        "RestingECG_Normal":                1 if inputs["RestingECG"] == "Normal" else 0,
        "RestingECG_ST":                    1 if inputs["RestingECG"] == "ST"     else 0,

        # ST_Slope one-hot
        "ST_Slope_Down":                    1 if inputs["STSlope"] == "Down" else 0,
        "ST_Slope_Flat":                    1 if inputs["STSlope"] == "Flat" else 0,
        "ST_Slope_Up":                      1 if inputs["STSlope"] == "Up"   else 0,

        # Chol_category one-hot
        "Chol_category_Desirable":          1 if chol_cat == "Desirable"       else 0,
        "Chol_category_Borderline High":    1 if chol_cat == "Borderline High" else 0,
        "Chol_category_High":               1 if chol_cat == "High"            else 0,
    }

    df = pd.DataFrame([encoded])

    # Step 4: Apply same scalers from Phase 1 
    df["Age"]       = age_scaler.transform(df[["Age"]])
    df["RestingBP"] = resting_bp_scaler.transform(df[["RestingBP"]])
    df["MaxHR"]     = max_hr_scaler.transform(df[["MaxHR"]])
    df["Oldpeak"]   = oldpeak_scaler.transform(df[["Oldpeak"]])

    result = model.predict(df)
    return "Heart Disease" if result[0] == 1 else "No Heart Disease"

In [5]:
#JUST AN EXAMPLE, EACH ONE OF US WILL DO LIKE THIS WHEN HER PROMPT IS READY, SHE WILL PREPARE THE INPUT, THEN SEND TO OUR MODEL TO GET THE 
#PREDICTION THAT WILL BE SEND WITHIN THE PROMPT
inputs = {
    "Age":              55,
    "Sex":              "M",
    "ChestPainType":  "ATA",
    "RestingBP":       140,
    "Cholesterol":      240,
    "FastingBS":       1,
    "RestingECG":      "Normal",
    "MaxHR":           150,
    "ExerciseAngina":  "Y",
    "Oldpeak":          2.3,
    "STSlope":         "Flat"
}

inputs["prediction"] = get_prediction(inputs)
print(inputs["prediction"])

Heart Disease


### Generative AI setup

In [169]:
#Imports & Client Setup
import os
from google import genai
from dotenv import load_dotenv, find_dotenv

load_dotenv(override=True)  # override=True forces it to reload
api_key = os.getenv("GEMINI_API_KEY")
print(api_key[:15])  # confirm it's the new key
client = genai.Client(api_key=api_key)
MODEL = "gemini-2.5-flash"

AQ.Ab8RN6Kw5yAK


In [170]:
#Shared Method 
def call_gemini(system_prompt, user_prompt):
    response = client.models.generate_content(
        model=MODEL,
        contents=user_prompt,
        config={"system_instruction": system_prompt}
    )
    return response.text

In [10]:
#JUST AN EXAMPLE, EACH ONE OF US WILL DO LIKE THIS WHEN HER PROMPT IS READY, SHE WILL PREPARE THE INPUT, THE USER PROMPT, THE SYSTEM PROMPT
#THEN CALL THIS METHOD THAT IS ALREADY PREPARED FOR YOU GUYS
inputs = {
    "prediction":       "No Heart Disease",
    "Age":              55,
    "Sex":              "Male",
    "ChestPainType":  "ATA",
    "RestingBP":       140,
    "Cholesterol":      240,
    "FastingBS":       1,
    "RestingECG":      "Normal",
    "MaxHR":           150,
    "ExerciseAngina":  "Yes",
    "Oldpeak":          2.3,
    "STSlope":         "Flat"
}

SYSTEM_T1 = """You are a medical assistant that helps explain heart disease risk to patients."""

USER_T1 = """A patient has been assessed and the result is: {prediction}.
Patient details: Age={Age}, Sex={Sex}, Chest Pain={ChestPainType}, Resting BP={RestingBP},
Cholesterol={Cholesterol}, Fasting BS={FastingBS}, Resting ECG={RestingECG},
Max HR={MaxHR}, Exercise Angina={ExerciseAngina}, Oldpeak={Oldpeak}, ST Slope={STSlope}.
Please provide a brief and simple explanation of this result."""

result_t1 = call_gemini(SYSTEM_T1, USER_T1.format(**inputs))
print(result_t1)

KeyboardInterrupt: 

# Generated Knowledge Prompting

 ### Template ID 
 1

---
### Approach Definition

Generated Knowledge Prompting is a prompt engineering technique introduced
by Liu et al. (2022) that instructs the model to first generate relevant
knowledge about the topic before producing its final response. Rather than
asking the model to answer directly, the prompt is structured in two
internal stages within a single request: the model first surfaces and
organises what it knows about the subject, and then uses that
self-generated knowledge as the foundation for its final output.

The core idea behind this technique is that large language models contain
vast amounts of knowledge from their training, but do not always surface
the most relevant information when asked to respond directly. By explicitly
instructing the model to generate knowledge first, the prompt forces a
broader and more deliberate retrieval of relevant information before any
conclusion is produced. This generated knowledge then sits within the
model's context during the response generation phase, ensuring that the final
response is grounded in a richer and more organised understanding of the
topic (Liu et al., 2022). This approach does not require access to an
external knowledge base or any task-specific training, making it flexible
and practical for a wide range of applications.

> Liu, J., Liu, A., Lu, X., Welleck, S., West, P., Le Bras, R., Choi, Y.,
> and Hajishirzi, H. (2022). *Generated Knowledge Prompting for Commonsense
> Reasoning.* Proceedings of the 60th Annual Meeting of the Association for
> Computational Linguistics, 3154–3169.
> https://doi.org/10.18653/v1/2022.acl-long.225


---

### Relation to Medical Field

In a medical context, generating knowledge before producing a response is
particularly valuable because clinical reasoning is rarely a direct
input-to-output process. A clinician examining a patient's cardiovascular
data does not immediately jump to a conclusion. They first consider what
each measurement means, how it relates to known clinical patterns, and what
risks it might indicate, before forming an assessment and recommending
next steps. Generated Knowledge Prompting mirrors this interpretive process
by instructing the model to first surface and organise its clinical
knowledge about the patient's specific feature values before producing
the final advisory.

This is especially important in cardiovascular health, where the clinical
significance of a measurement depends heavily on context. A resting blood
pressure of 145 mmHg, for example, cannot be interpreted in isolation. The
model needs to draw on its knowledge of what that value means, how it
relates to cardiovascular risk, and how it interacts with other features
in the patient's profile before it can generate advice that is genuinely
meaningful. By explicitly instructing the model to generate this contextual
knowledge first, the prompt ensures that the final advisory is grounded in
a deliberate clinical reasoning step rather than a surface level response.


---


### Intended Use Case

 **1- Deployment Environment**

The system is planned to be deployed as an advisory tool within a hospital
mobile application. Patients access it from their personal smartphones
after obtaining their clinical data, either from a recent cardiovascular
test or by completing one at the hospital. The patient enters their
clinical values into the tool and receives a personalized advisory
directly on the same screen.

 **2- The User**

The primary user of this template is a **first-time patient** who is
encountering the system and their clinical results for the very first
time. This patient arrives with no frame of reference for what their
numbers mean. A resting blood pressure of 145 mmHg is just a number to
them. An Oldpeak value of 2.3 carries no weight without context. A
cholesterol category label means nothing without knowing what it implies
for their health. Providing advice to this patient without first
establishing what their values mean would produce a response they cannot
connect to their own situation, and advice that cannot be connected to
is advice that will not be acted on. Research in health education
confirms that patients who understand the meaning of their clinical
measurements are significantly more likely to follow through on health
recommendations than those who receive advice without explanation
(Berkman et al., 2011).

< Berkman, N.D., et al. (2011). *Health literacy interventions and
> outcomes: An updated systematic review.* Evidence Report/Technology
> Assessment, 199, 1–941.

***Medical background:*** No medical background is assumed. The gap
between what the patient's values say and what they mean to that patient
is treated as the central design constraint of this template. Before any
recommendation can land meaningfully, the patient needs enough
foundational understanding to connect the advice back to their own
situation rather than receiving it as an instruction they must simply
trust.

***Emotional state:*** First-time users often arrive uncertain and quietly
apprehensive. They may not know what they are looking for or what a
concerning result would even look like. The tone of the advisory must
therefore be steady and educational rather than clinical or alarming.
The goal is not to reassure unconditionally or to alarm unnecessarily,
but to leave the patient feeling genuinely informed and oriented enough
to take the next right step.

**3- Inputs**

3.1- The patient heart disease classification result 

3.2- The patient clinical feature values


**4- Output**

A single advisory paragraph written in plain, accessible language.
Although the model generates clinical knowledge internally as part of
its knowledge generation process, none of this internal knowledge generation is
visible to the patient. The output reads as a clean, direct advisory,
but because it was built on top of a deliberate knowledge generation
step, it is more grounded in the patient's specific values and more
educationally coherent than a direct response would be for someone
encountering their clinical data for the first time. The patient leaves
the interaction not only with advice to follow but with enough implicit
understanding of their values to begin asking the right questions at
their next clinical appointment.


---

### Design Rationale

**Advantage 1 — Grounded knowledge generation before response:**
Generated Knowledge Prompting instructs the model to first surface and
organise relevant knowledge about the input before producing its final
response, ensuring the output is grounded in an explicit knowledge generation step
rather than a direct leap from input to answer (Liu et al., 2022). In a
heart disease advisory system, this matters because the clinical
significance of a patient's feature values cannot be determined without
first interpreting what those values mean in a cardiovascular context.
A model that jumps directly to advice without first generating knowledge about the
patient's specific measurements risks producing generic or misaligned
recommendations that do not reflect the patient's actual profile.

**Advantage 2 — Improved contextual relevance for individual patients:**
By generating knowledge that is specific to the input provided, the
technique ensures that the final response is tailored to the individual
patient rather than relying on broad generalisations (Liu et al., 2022).
In a heart disease advisory system where two patients can share the same
prediction label but have entirely different clinical profiles, this
matters directly. A patient with a borderline resting blood pressure and
a flat ST slope requires fundamentally different advice from a patient
with a normal blood pressure and an asymptomatic chest pain type, even
if both are classified as No Heart Disease Detected. Generated Knowledge Prompting ensures the model generates contextual
knowledge about the specific values present before writing the advisory,
producing a response that reflects the individual rather than the label.

**Advantage 3 — No dependency on predefined examples:**
Generated Knowledge Prompting does not require carefully crafted
input-output examples to guide the model's behaviour. The model draws
on its own clinical knowledge to reason about the patient's data, making
the technique flexible and robust across the full range of possible
patient profiles (Liu et al., 2022). In a heart disease advisory system
where patient feature values vary continuously across a wide range,
providing representative examples that cover all meaningful combinations
is not practical. Generated Knowledge Prompting eliminates this
dependency by allowing the model to generate the contextual knowledge
it needs dynamically from the input itself.


> Liu, J., Liu, A., Lu, X., Welleck, S., West, P., Le Bras, R., Choi, Y.,
> and Hajishirzi, H. (2022). *Generated Knowledge Prompting for Commonsense
> Reasoning.* Proceedings of the 60th Annual Meeting of the Association for
> Computational Linguistics, 3154–3169.
> https://doi.org/10.18653/v1/2022.acl-long.225

### Prompt Engineering Techniques Applied
***1. Role Assignment***
The system instruction opens by assigning the model a specific professional
role as a clinical decision-support assistant specialised in cardiovascular
health. This role anchors the model's tone, vocabulary, and level of
clinical awareness for every response. By defining the role explicitly,
the model is prevented from responding as a general-purpose assistant and
is instead constrained to operate within a cardiovascular health context
that is appropriate for the system's purpose (Google, 2024).

Applied in the system instruction:
**You are a clinical decision-support assistant specialised in
cardiovascular health.**

> Google. (2024). *Introduction to prompting.*
> Google AI for Developers.
> https://ai.google.dev/gemini-api/docs/prompting-intro

***2. User Context Priming***
The system instruction explicitly describes the target user as a first-time
patient with no frame of reference for their clinical values. This context
primes the model to calibrate the depth, length, and tone of its response
appropriately. Without this context, the model may produce responses that
are either too technical for a first-time patient or too generic to be
meaningful (Cascella et al., 2024).

Applied in the system instruction:
**The patient receiving this insight is a first-time user who has never seen
or interpreted their clinical data before. They have no frame of reference
for what their values mean.**

> Cascella, M., Montomoli, J., Bellini, V. and Bignami, E. (2024).
> *Prompt Engineering in Healthcare.* Electronics, 13(15), 2961.
> https://doi.org/10.3390/electronics13152961


***3. Conciseness Constraint***
The system instruction explicitly instructs the model not to over-explain
and to focus only on the most relevant values. This constraint is
particularly important for a first-time patient who may feel overwhelmed
by too much information. Keeping the output concise and focused ensures
the patient leaves the interaction feeling informed and oriented rather
than confused (Cascella et al., 2024).

Applied in the system instruction:
**Do not over-explain. Do not list every value. Focus only on the most
relevant values and explain them briefly and clearly. The goal is to
leave the patient feeling informed and oriented, not overwhelmed.**

> Cascella, M., Montomoli, J., Bellini, V. and Bignami, E. (2024).
> *Prompt Engineering in Healthcare.* Electronics, 13(15), 2961.
> https://doi.org/10.3390/electronics13152961

***4. Tone Specification***
The system instruction explicitly specifies the tone the model must
maintain throughout the response: calm, grounding, and educational. In
a medical context, tone is as important as content. A response that is
accurate but alarming or cold can cause unnecessary distress for a
first-time patient. By specifying the tone explicitly, the prompt ensures
the model communicates in a way that is appropriate for someone
encountering their clinical data for the very first time (Gharib et al.,
2024).

Applied in the system instruction:
**Use a calm, grounding, and educational tone.**

> Gharib, A.M., et al. (2024). *A guide to prompt design: Foundations
> and applications for healthcare simulationists.* Frontiers in Medicine,
> 11, 1504532.
> https://doi.org/10.3389/fmed.2024.1504532

***5. Value Grounding***
The system instruction explicitly instructs the model to ground its
response in the patient's specific clinical values rather than generic
disease labels. This ensures that every patient receives a response that
reflects their individual numbers, not a template response triggered by
the prediction label alone. This is essential for personalisation and
is directly aligned with the objective of the system (Liu et al., 2022).

Applied in the system instruction:
**Be grounded in the patient's specific values, not in generic disease
labels.**

> Cascella, M., Montomoli, J., Bellini, V. and Bignami, E. (2024).
> *Prompt Engineering in Healthcare.* Electronics, 13(15), 2961.
> https://doi.org/10.3390/electronics13152961


***6. Conditional Response Logic***
The system instruction defines three distinct response scenarios based
on the prediction label and individual feature values: Heart Disease
Detected, No Heart Disease with all healthy values, and No Heart Disease
with borderline values. Each scenario is given specific and detailed
instructions for how the model should respond, ensuring the output is
always contextually appropriate regardless of which scenario the patient
falls into.

Applied in the system instruction:

**- If the result is Heart Disease Detected: focus on explaining which
    values are contributing to the result and what they mean**

**- If the result is No Heart Disease Detected and all values are healthy:
    reassure the patient and explain which specific values are within
    healthy ranges and what each one means for their cardiovascular health**

**- If the result is No Heart Disease Detected but one or more values are
    borderline: acknowledge the assessment result but explain the borderline
    values and what they could mean if left unaddressed**

 ***7. Safety Constraints***
The system instruction includes explicit safety constraints that the model
must follow in every response. The model must never state or imply a
diagnosis, must never use clinical jargon without explanation, and must
only recommend consulting a medical professional when it is clinically
necessary. These constraints ensure the system operates safely and
responsibly within a medical context where an inappropriate response
could directly impact patient trust and wellbeing (Cascella et al., 2024).

Applied in the system instruction:

**- Never use clinical jargon without explanation**

**- Never state or imply a diagnosis. Always refer to the assessment
    result or the clinical indicators instead.**

**- Recommend consulting a medical professional only when it is clinically
    necessary.**

### Prompt Structure
**System Instruction:**

You are a clinical decision-support assistant specialised in cardiovascular health. 
You communicate in plain, accessible language suitable for patients with no medical background.

The patient receiving this insight is a first-time user who has never seen 
or interpreted their clinical data before. They have no frame of reference 
for what their values mean. Your insight must therefore be concise and 
focused. Do not over-explain. Do not list every value. Focus only on the 
most relevant values and explain them briefly and clearly. The goal is to 
leave the patient feeling informed and oriented, not overwhelmed.

You follow a two-stage internal process for every patient assessment:

STAGE 1 — GENERATE CLINICAL KNOWLEDGE (INTERNAL ONLY — DO NOT SHOW TO PATIENT):
Before writing any insight, you must first internally generate your clinical 
knowledge about each of the patient's values. For each feature provided, establish 
what that value means, whether it is healthy, borderline, or concerning, and what 
its implications are for cardiovascular health. This stage must never appear in 
your response. It is a silent knowledge generation step only.

STAGE 2 — WRITE THE INSIGHT (THIS IS THE ONLY THING YOU OUTPUT):
Using the knowledge you generated in Stage 1, write a structured personalised
insight for the patient. The insight must follow this exact format:

[One sentence introducing the assessment result and what this insight will cover.]

[One short paragraph explaining the most relevant values and what they indicate.
Focus only on concerning or borderline values. Keep it brief and clear.]

[One closing sentence encouraging the patient to consult a medical professional.]

The insight must also follow these rules:
- The core purpose is interpretation, not recommendation
- Be grounded in the patient's specific values, not in generic disease labels
- If the result is Heart Disease Detected: focus on explaining which values
  are contributing to the result and what they mean
- If the result is No Heart Disease Detected and all values are healthy:
  reassure the patient and explain which specific values are within healthy
  ranges and what each one means for their cardiovascular health. Help them
  understand why their results are reassuring, not just that they are.
- If the result is No Heart Disease Detected but one or more values are borderline:
  acknowledge the assessment result but explain the borderline values and what
  they could mean if left unaddressed
- Use a calm, grounding, and educational tone
- Never use clinical jargon without explanation
- Never state or imply a diagnosis. Always refer to the assessment result
  or the clinical indicators instead.
- Recommend consulting a medical professional only when it is clinically
  necessary, such as when values are concerning or borderline. Do not
  include this recommendation if all values are healthy and reassuring.
- NO headers, NO bullet points, NO section titles, NO numbered sections
- Each of the three parts must be separated by a blank line for readability

YOUR RESPONSE MUST CONTAIN ONLY THREE SHORT PARAGRAPHS SEPARATED BY BLANK LINES.
NOTHING ELSE.

**User Prompt:**

Patient Assessment Result: {prediction}

Patient Clinical Values:
- Age: {Age} years
- Sex: {Sex}
- Chest Pain Type: {ChestPainType}
- Resting Blood Pressure: {RestingBP} mmHg
- Cholesterol: {Cholesterol} mg/dL
- Fasting Blood Sugar above 120 mg/dL: {FastingBS} (0 = No, 1 = Yes)
- Resting ECG: {RestingECG}
- Maximum Heart Rate: {MaxHR} bpm
- Exercise-Induced Angina: {ExerciseAngina}
- Oldpeak (ST Depression): {Oldpeak}
- ST Slope: {STSlope}

First generate your internal clinical knowledge about each feature silently,
then write the structured personalised insight for this patient
following the exact three paragraph format specified.

### Input / Output Example

#### System and User Prompt

In [24]:
SYSTEM_T1 = """You are a clinical decision-support assistant specialised in cardiovascular health. 
You communicate in plain, accessible language suitable for patients with no medical background.

The patient receiving this insight is a first-time user who has never seen 
or interpreted their clinical data before. They have no frame of reference 
for what their values mean. Your insight must therefore be concise and 
focused. Do not over-explain. Do not list every value. Focus only on the 
most relevant values and explain them briefly and clearly. The goal is to 
leave the patient feeling informed and oriented, not overwhelmed.

You follow a two-stage internal process for every patient assessment:

STAGE 1 — GENERATE CLINICAL KNOWLEDGE (INTERNAL ONLY — DO NOT SHOW TO PATIENT):
Before writing any insight, you must first internally generate your clinical 
knowledge about each of the patient's values. For each feature provided, establish 
what that value means, whether it is healthy, borderline, or concerning, and what 
its implications are for cardiovascular health. This stage must never appear in 
your response. It is a silent knowledge generation step only.

STAGE 2 — WRITE THE INSIGHT (THIS IS THE ONLY THING YOU OUTPUT):
Using the knowledge you generated in Stage 1, write a structured personalised
insight for the patient. The insight must follow this exact format:

[One sentence introducing the assessment result and what this insight will cover.]

[One short paragraph explaining the most relevant values and what they indicate.
Focus only on concerning or borderline values. Keep it brief and clear.]

[One closing sentence encouraging the patient to consult a medical professional.]

The insight must also follow these rules:
- The core purpose is interpretation, not recommendation
- Be grounded in the patient's specific values, not in generic disease labels
- If the result is Heart Disease Detected: focus on explaining which values
  are contributing to the result and what they mean
- If the result is No Heart Disease Detected and all values are healthy:
  reassure the patient and explain which specific values are within healthy
  ranges and what each one means for their cardiovascular health. Help them
  understand why their results are reassuring, not just that they are.
- If the result is No Heart Disease Detected but one or more values are borderline:
  acknowledge the assessment result but explain the borderline values and what
  they could mean if left unaddressed
- Use a calm, grounding, and educational tone
- Never use clinical jargon without explanation
- Never state or imply a diagnosis. Always refer to the assessment result
  or the clinical indicators instead.
- Recommend consulting a medical professional only when it is clinically
  necessary, such as when values are concerning or borderline. Do not
  include this recommendation if all values are healthy and reassuring.
- NO headers, NO bullet points, NO section titles, NO numbered sections
- Each of the three parts must be separated by a blank line for readability

YOUR RESPONSE MUST CONTAIN ONLY THREE SHORT PARAGRAPHS SEPARATED BY BLANK LINES.
NOTHING ELSE."""


USER_T1 = """Patient Assessment Result: {prediction}

Patient Clinical Values:
- Age: {Age} years
- Sex: {Sex}
- Chest Pain Type: {ChestPainType}
- Resting Blood Pressure: {RestingBP} mmHg
- Cholesterol: {Cholesterol} mg/dL
- Fasting Blood Sugar above 120 mg/dL: {FastingBS} (0 = No, 1 = Yes)
- Resting ECG: {RestingECG}
- Maximum Heart Rate: {MaxHR} bpm
- Exercise-Induced Angina: {ExerciseAngina}
- Oldpeak (ST Depression): {Oldpeak}
- ST Slope: {STSlope}

First generate your internal clinical knowledge about each feature silently,
then write the structured personalised insight for this patient
following the exact three paragraph format specified."""

#### Test case 1

In [25]:
# Test Case 1: Heart Disease Detected
inputs_1 = {
    "Age":              55,
    "Sex":              "Male",
    "ChestPainType":    "ATA",
    "RestingBP":        140,
    "Cholesterol":      240,
    "FastingBS":        1,
    "RestingECG":       "Normal",
    "MaxHR":            150,
    "ExerciseAngina":   "Yes",
    "Oldpeak":          2.3,
    "STSlope":          "Flat"
}
inputs_1["prediction"] = get_prediction(inputs_1)

from IPython.display import display, Markdown
print("Test Case 1 — Heart Disease Detected")
result_1 = call_gemini(SYSTEM_T1, USER_T1.format(**inputs_1))
display(Markdown(result_1))


Test Case 1 — Heart Disease Detected


Your recent assessment indicates the presence of heart disease, and this insight will help you understand what your results mean.

Several clinical indicators contributed to this assessment, including your elevated blood pressure of 140 mmHg, which is considered high, and your total cholesterol level of 240 mg/dL, which is also high. Additionally, your fasting blood sugar was found to be above 120 mg/dL, indicating high blood sugar. You also reported chest pain with exercise, and your exercise ECG showed significant changes (ST depression of 2.3 with a flat slope), which can indicate that your heart muscle is not getting enough blood flow during physical activity.

Given these findings, it is important to discuss these results with your doctor or a healthcare professional who can provide personalised advice and guidance on the next steps.

#### Test case 2

In [26]:
# Test Case 2: No Heart Disease Detected — All Healthy Values
inputs_2 = {
    "Age":              35,
    "Sex":              "Female",
    "ChestPainType":    "NAP",
    "RestingBP":        110,
    "Cholesterol":      180,
    "FastingBS":        0,
    "RestingECG":       "Normal",
    "MaxHR":            175,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.0,
    "STSlope":          "Up"
}
inputs_2["prediction"] = get_prediction(inputs_2)

from IPython.display import display, Markdown
print("Test Case 2 — No Heart Disease: All Healthy Values")
result_2 = call_gemini(SYSTEM_T1, USER_T1.format(**inputs_2))
display(Markdown(result_2))

Test Case 2 — No Heart Disease: All Healthy Values


Your recent assessment indicates no heart disease, and this insight will help you understand what your results mean for your cardiovascular health.

This is reassuring news, as several key indicators of heart health are well within healthy ranges. For example, your resting blood pressure of 110 mmHg and your cholesterol level of 180 mg/dL are both at healthy levels, meaning your heart isn't working harder than it should and your arteries are likely clear. Additionally, your blood sugar is not elevated, and your heart's response to exercise, including your maximum heart rate and the absence of any concerning changes on your ECG during exertion, shows good fitness and no signs of concern.

It's encouraging to see such healthy results, which suggest your cardiovascular system is currently in good shape.

#### Test case 3

In [27]:

# Test Case 3: No Heart Disease Detected — Borderline Values
inputs_3 = {
    "Age":              45,
    "Sex":              "Male",
    "ChestPainType":    "NAP",
    "RestingBP":        128,
    "Cholesterol":      210,
    "FastingBS":        0,
    "RestingECG":       "Normal",
    "MaxHR":            160,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.8,
    "STSlope":          "Up"
}
inputs_3["prediction"] = get_prediction(inputs_3)


from IPython.display import display, Markdown
print("Test Case 3 — No Heart Disease: Borderline Values")
result_3 = call_gemini(SYSTEM_T1, USER_T1.format(**inputs_3))
display(Markdown(result_3))

Test Case 3 — No Heart Disease: Borderline Values


This assessment provides an overview of your heart health, indicating no heart disease detected at this time, but highlighting some important points about your clinical values.

While your overall assessment is reassuring, some of your values are at levels that warrant attention. Your resting blood pressure of 128 mmHg is slightly elevated, meaning your heart is working a bit harder than ideal, and your total cholesterol level of 210 mg/dL is borderline high. Additionally, during your exercise test, a measurement called 'Oldpeak' showed a slight change (0.8), which can sometimes indicate that your heart may not be getting quite enough blood flow during physical exertion.

These findings provide important insights, and we recommend discussing them with your doctor for personalised advice on maintaining your heart health.

# Chain-of-Thought Prompting

### Template ID 
2

---

### Approach Definition

Chain-of-thought (CoT) prompting is a prompt engineering technique in which a language model is instructed to produce a series of intermediate reasoning steps before arriving at a final answer. Rather than mapping an input directly to an output in a single pass, the model externalizes its thinking process, each step builds on the previous one, and the final answer is explicitly grounded in the visible reasoning chain. IBM describes CoT as a technique that simulates human-like reasoning by breaking down elaborate problems into manageable, intermediate steps that sequentially lead to a conclusive answer (IBM, 2024). The technique is activated by including a structured reasoning directive in the prompt, most commonly the phrase "think step by step," which signals to the model that intermediate steps are required as part of the response rather than just a final conclusion.


> IBM. (2024). *What is chain of thought (CoT) prompting?* IBM Think.  
> __https://www.ibm.com/think/topics/chain-of-thoughts__

---

### Relation to Medical Field

The structure of chain-of-thought prompting mirrors the way clinicians think when making a diagnosis. A doctor assessing a patient for cardiovascular disease does not jump straight from test results to a treatment decision. Patel and Groen (1986) studied how cardiologists reason through complex cases and found that accurate diagnoses were reached by moving in one direction only: from clinical findings, step by step, toward a conclusion. Each step followed from the one before it, building on the available evidence rather than starting from an assumed answer. This is exactly how chain-of-thought prompting works: the model moves forward through a series of reasoning steps, each grounded in the previous one, without going back to revise earlier conclusions, making CoT a naturally suitable technique for clinical decision-making tasks.

> Patel, V. L., & Groen, G. J. (1986). Knowledge based solution strategies in medical reasoning. *Cognitive Science*, *10*(1), 91–116. 
> https://doi.org/10.1207/s15516709cog1001_4
---

### Intended Use Case

**1- Deployment Environment**

The system is planned to be deployed as an advisory tool within a hospital
mobile application. Patients access it from their personal smartphones
after obtaining their clinical data, either from a recent cardiovascular
test or by completing one at the hospital. The patient enters their
clinical values into the tool and receives a personalized advisory
directly on the same screen. 

**2- The User**

The primary user of this template is a detail-oriented patient who approaches their health result with curiosity and a desire to understand rather than simply act. This patient is likely to be someone who has prior experience with cardiovascular health (either through a personal history of monitoring their health closely, a family member with heart disease, or a general habit of researching their medical information). They are comfortable reading a longer, structured response and will invest the time to follow a step-by-step explanation from beginning to conclusion. 
Research in patient health literacy confirms that a significant proportion of patients actively prefer detailed explanations of their results, as understanding the reasoning behind a recommendation increases their confidence in acting on it and their trust in the system delivering it (Zolnierek and Dimatteo, 2009).
> Zolnierek, K.B.H. and DiMatteo, M.R. (2009). *Physician communication
> and patient adherence to treatment: A meta-analysis.* Medical Care,
> 47(8), 826–834.
> https://doi.org/10.1097/MLR.0b013e31819a5acc

***Medical background:*** While no formal medical background is assumed, this patient is health-literate. They are capable of understanding concepts such as blood pressure ranges, cholesterol categories, and the general relationship between lifestyle factors and cardiovascular risk. The advisory may therefore use slightly more specific language than the zero-shot template, as long as each term is explained in context. (Is it okay to compare here?)

***Emotional state:*** This patient is engaged rather than acutely anxious. They are seeking information and understanding, which means a longer, more structured response will not increase their distress. However, the tone must still remain calm and professional. The visibility of the reasoning steps must never make the output feel like a clinical verdict.

**3- Inputs**

3.1 The patient heart disease classification result 

3.2 The patient clinical feature values


**4- Output**

A structured response consisting of a brief per-feature assessment followed by a synthesised final advisory paragraph. The reasoning steps walk through the patient's most significant clinical values one by one, identifying what each means and how it contributes to the overall picture. The paragraph then synthesises these steps into a coherent, actionable recommendation. The visible reasoning is written in plain language throughout so that the patient can follow each step without a medical background. Scrolling is acceptable here because the patient prefers this level of details. 

---

### Design Rationale

**Advantage 1 — Improved accuracy on complex reasoning tasks:**
Chain-of-thought improves model performance on complex tasks by breaking
them down into smaller, manageable steps, with each intermediate step
acting as a checkpoint where errors can be caught before they carry
forward into the final output (DataCamp, 2024). In a heart disease
advisory system, this matters because an error made early (such as
misreading a borderline cholesterol value or misclassifying a resting
ECG result) will corrupt every reasoning step that follows it. By
making each step visible and sequential, chain-of-thought allows such
errors to be identified at the point they occur rather than arriving
silently embedded in a final recommendation.

**Advantage 2 — Transparency and explainability:** By generating visible
intermediate reasoning steps, chain-of-thought makes the decision-making
process auditable and easier to follow for non-expert users (IBM, 2024).
In a heart disease advisory system where the intended users have no
medical background, this is a functional requirement. A patient who
receives only a label or a conclusion cannot understand what it
means for them personally, identify which of their values drove the
outcome, or communicate the basis of the advisory to a doctor. By
walking through each feature in plain, step-by-step reasoning before
reaching a conclusion, chain-of-thought produces an output that a
non-specialist can follow, question, and act on, rather than simply
accept or dismiss.

**Advantage 3 — Multistep reasoning capability:** Chain-of-thought
handles tasks that require multistep reasoning by systematically working
through each component of a problem before reaching a conclusion, leading
to more reliable outputs (IBM, 2024). In cardiovascular test,
reliability is critical where a premature conclusion or a skipped step can
result in an advisory that does not reflect the patient's actual
condition. By structuring the output as a sequence of dependent reasoning
steps, chain-of-thought ensures the model cannot arrive at a conclusion
before all components of the problem have been addressed, making the
final output consistent and reproducible regardless of how the input
values are distributed.

**Advantage 4 — Attention to detail:** The step-by-step nature of
chain-of-thought encourages detailed breakdowns of complex inputs,
ensuring each component is examined and accounted for before a conclusion
is reached (IBM, 2024). In cardiovascular test, where no
single factor is decisive on its own, this level of detail is essential.
The ACC/AHA Pooled Cohort Equations calculate cardiovascular risk from
multiple variables whose combined effect is non-additive, meaning the
contribution of any one feature depends on the values of the others
(Goff et al., 2014). Chain-of-thought ensures that no feature is skipped
or treated as insignificant, making it the most detail-preserving
technique available for this task.


> DataCamp. (2024). *Chain-of-thought prompting*. 
> https://www.datacamp.com/tutorial/chain-of-thought-prompting

> IBM. (2024). *Chain of thoughts*. IBM Think. 
> https://www.ibm.com/think/topics/chain-of-thoughts

> Goff, D. C., Lloyd-Jones, D. M., Bennett, G., Coady, S., D'Agostino, R. B.,  Gibbons, R., Greenland, P., Lackland, D. T., Levy, D., O'Donnell, C. J.,  Robinson, J. G., Schwartz, J. S., Shero, S. T., Smith, S. C., Sorlie, P., Stone, N. J., & Wilson, P. W. F. (2014). 2013 ACC/AHA guideline on the assessment of cardiovascular risk: A report of the American College of Cardiology/American Heart Association Task Force on Practice Guidelines. *Journal of the American College of Cardiology*, *63*(25), 2935–2959. 
> https://doi.org/10.1016/j.jacc.2013.11.005

---
### Prompt Engineering Techniques Applied

#### System Prompt — Role

*You are a cardiovascular health analyst deployed in a hospital mobile application.*

The role assigned to the model is **cardiovascular health analyst**. This was chosen deliberately over alternatives like "health advisor" or "health educator." An advisor gives recommendations. An educator builds knowledge in someone who lacks it. An analyst examines evidence and makes reasoning visible step by step, which is exactly what this template requires for a detail-oriented patient who wants to follow the thinking behind their result, not just receive a conclusion.

#### System Prompt — Role Description

*Your role is to help health-literate patients understand the results of their cardiovascular test by examining each clinical value in turn, making your reasoning visible, and then delivering a final personalised advisory.*

This sentence defines the three-stage job the model must perform: **(1)** examine each value individually, **(2)** make the reasoning visible at each step, and **(3)** deliver a final advisory after all values have been analysed. The phrase **"making your reasoning visible"** is what activates chain-of-thought behaviour. It tells the model that intermediate reasoning must appear in the output and must not be hidden or summarized internally.

#### System Prompt — Patient Profile

*The patient reading this response is health-literate and engaged. They are not a medical professional, but they are comfortable with concepts such as blood pressure categories, cholesterol levels, and the relationship between lifestyle habits and cardiovascular risk. They expect each of their test values to be explained individually and connected to the overall picture before a recommendation is made.*

This paragraph calibrates the vocabulary level the model should write at. It tells the model two things simultaneously: do not over-simplify (this patient knows what cholesterol and blood pressure mean), and do not assume clinical expertise (they cannot interpret an Oldpeak reading or ST slope without explanation). 

#### Rule 1 — Visible Reasoning, No Structural Labels

*Always show your reasoning steps visibly in the output. Do NOT hide or summarise the steps internally. Present your analysis in a clear, logical flow — do not use any step labels, headers, or numbering such as "Step 1", "Step 2", "Part 1", "Analysis", "Advisory", or any similar structural label anywhere in your response.*

Two instructions are combined here. The first: always show reasoning visibly which is the core chain-of-thought requirement. The second: no structural labels, exists because the Python code at the end of this template splits the response using its own separator and adds its own labels when displaying the output. If the model adds headers like "Step 1" or "Advisory", they would appear in the wrong place in the final rendered output and confuse the patient.

#### Rule 2 — Per-Value Tables

*Begin your response by analysing each cardiovascular test value. For each value, present your findings in its own individual two-column table with the following rows: Test Value, What It Measures, Assessment, What It May Suggest.*

Each value gets its own small vertical two-column table. 

*Contextual values such as Age and Sex should have their own small table with only the Test Value and Assessment rows filled — leave What It Measures and What It May Suggest empty for those rows.*

Age and Sex are non-modifiable contextual factors: the patient cannot change them, and they do not directly drive lifestyle recommendations. Giving them the full four-row treatment would produce unnecessary output. This instruction keeps them brief while still acknowledging them as part of the analysis.

Since this is displayed within a hospital mobile application, screen space is limited and presenting the full per-value analysis inline would overwhelm the patient with text. The detailed tables can therefore be designed as supplementary content where in a complete system implementation, they would be exportable as a downloadable PDF that the patient can save, print, or bring to their next clinical appointment for a more thorough discussion with their doctor.

#### Rule 3 — The Advisory Separator and Three-Branch Logic

*After completing the table for every test value, insert this exact line on its own: ---ADVISORY---*

This separator is a technical requirement for the Python code at the bottom of the template. The `split("---ADVISORY---")` call uses it to divide the response into two parts: the tables and the advisory, so they can be displayed in reverse order. It must be exact and on its own line for the split to work reliably.

*If the prediction indicates heart disease is detected: identify the specific clinical values that are outside healthy ranges and generate advice tied directly to those values. If the prediction indicates no heart disease is detected and all values are within healthy ranges: reassure and reinforce the specific positive findings. If one or more values are borderline: warn specifically about each borderline value and advise the patient to monitor it.*

The advisory follows three-branch logic based on the combination of the prediction label and the actual feature values. This is important because two patients can share the same prediction label but have completely different clinical profiles. A patient classified as no heart disease with all healthy values needs a different response from a patient classified as no heart disease but with a borderline blood pressure reading. The three branches ensure the advisory is genuinely personalised rather than driven by the label alone.

#### Rule 4 — Advisory Length

*The advisory paragraph must be exactly 4 to 5 sentences.*

The length constraint applies only to the advisory paragraph, not to the tables. The tables can be as long as needed to cover all values thoroughly. The advisory is constrained because it is what will be displayed first on the mobile screen so must be immediately readable, a patient should not need to scroll just to reach the end of the summary.

#### Rule 5 — No Model References

*Never state a diagnosis and never reference the classification model directly. Do not use phrases like "the model predicts," "the classification result," or "the prediction."*

Stating a diagnosis is explicitly prohibited for safety reasons. This system is an AI-powered advisory tool, not a licensed clinical instrument. A diagnosis carries legal and medical weight that this system is not qualified to provide. It requires a trained physician who can examine the patient directly, review their full medical history, order additional tests, and account for factors that a structured dataset cannot capture. Presenting the model's output as a confirmed diagnosis could lead the patient to stop seeking professional care, act on incorrect information, or experience serious psychological harm from a conclusion that may not be accurate. Framing all findings as possibilities using language such as "may suggest" or "could indicate" is therefore not just a stylistic choice, it is a fundamental safety boundary that must be maintained across every response regardless of how clear or strong the clinical indicators appear to be.

#### Rule 6 — Plain Language with Explained Terms

*Write in plain, clear language. You may use health-literacy-level terms but always explain each term immediately after using it in one plain sentence.*

The model is permitted to use terms a patient would encounter on a medical information website. But every such term must be followed immediately by a plain-language explanation. This keeps the response accessible without being condescending to a health-literate reader.

#### Rule 7 — Tone

*Your tone must be calm, informative, and respectful. Do not be alarming, but do not soften findings to the point of obscuring their significance.*

This template prioritizes honest clarity over emotional reassurance. The intended users want a transparent analysis, where over-softening findings would frustrate them and defeat the purpose of visible reasoning.

#### Rule 8 — No Generic Advice

*Every recommendation or warning must name the specific test value it addresses and explain directly why that value makes this recommendation necessary. Generic advice with no traceable link to a specific finding is not acceptable.*

Every recommendation in the advisory must name a value, state what is concerning about it, and explain why the recommended action addresses it specifically.

#### User Prompt

The user prompt is intentionally minimal. It carries only the patient data and a single trigger sentence pointing back to the system prompt instructions. No rules or structure are repeated here keeping all logic in the system prompt avoids the risk of conflicting instructions between the two layers, which can cause unpredictable model behaviour.

---

### Prompt Structure

In [184]:
SYSTEM_T2 = """You are a cardiovascular health analyst deployed in a hospital \
mobile application. Your role is to help health-literate patients understand \
the results of their cardiovascular test by examining each clinical value in \
turn, making your reasoning visible, and then delivering a final personalised \
advisory.

The patient reading this response is health-literate and engaged. They are \
not a medical professional, but they are comfortable with concepts such as \
blood pressure categories, cholesterol levels, and the relationship between \
lifestyle habits and cardiovascular risk. They expect each of their test \
values to be explained individually and connected to the overall picture \
before a recommendation is made.

Follow these rules at all times:
1. Always show your reasoning steps visibly in the output. Do NOT hide or \
   summarise the steps internally. Present your analysis in a clear, logical \
   flow — do not use any step labels, headers, or numbering such as \
   "Step 1", "Step 2", "Part 1", "Analysis", "Advisory", or any similar \
   structural label anywhere in your response.

2. Begin your response by analysing each cardiovascular test value. For each \
   value, present your findings in its own individual two-column table with \
   the following rows:
   | Test Value          | [value name and number] |
   | What It Measures    | [one sentence]          |
   | Assessment          | [one sentence]          |
   | What It May Suggest | [one sentence]          |
   
   Contextual values such as Age and Sex should have their \
   own small table with only the Test Value and Assessment rows filled — \
   leave What It Measures and What It May Suggest empty for those rows. \
   Reserve the full four-row table for clinically actionable values only — \
   those that can directly inform lifestyle recommendations or signal \
   cardiovascular risk.
   
   Contextual values such as Age and Sex should follow the same format but \
   with only the Assessment bullet point filled — omit What It Measures and \
   What It May Suggest for those. Reserve the full three-bullet format for \
   clinically actionable values only — those that can directly inform \
   lifestyle recommendations or signal cardiovascular risk.

3. After completing the table for every test value, insert this exact line \
   on its own with nothing before or after it on that line:
   ---ADVISORY---
   Then, immediately after that line, write a single coherent \
   advisory paragraph. The content of this paragraph must be determined by \
   the prediction and the individual clinical values, as follows:

   If the prediction indicates heart disease is detected: identify the \
   specific clinical values that are outside healthy ranges and are \
   contributing to the risk. Generate advice tied directly to those specific \
   values. 

   If the prediction indicates no heart disease is detected, apply one of \
   two approaches based on the actual clinical values:

   First, if all feature values are within healthy ranges: acknowledge the \
   healthy status clearly and encourage the patient to maintain their current \
   lifestyle and habits, reinforcing the specific positive findings that \
   support this outcome.

   Second, if one or more feature values are borderline — meaning they are \
   technically within the healthy classification range but are approaching \
   concerning thresholds — generate a targeted warning for each such feature. \
   Name the specific value, explain why it is approaching a concerning \
   threshold, and advise the patient to monitor it and take preventive action \
   before it progresses. Do not simply reassure a patient whose values are \
   borderline. This advisory must be genuinely personalised even when the \
   patient is currently classified as healthy.

4. The advisory paragraph must be exactly 4 to 5 sentences. Acknowledge the \
   overall finding in the first sentence. Use the next 1 to 2 sentences to \
   address the most significant findings with specific recommendations or \
   warnings tied directly to the named clinical values. 

5. Never state a diagnosis and never reference the classification model \
   directly in your output. Do not use phrases like "the model predicts," \
   "the classification result," or "the prediction." Present the overall \
   finding naturally as part of your analysis without labelling its source.

6. Write in plain, clear language throughout. You may use health-literacy-level \
   terms but always explain what each term \
   means immediately after using it in one plain sentence.

7. Your tone must be calm, informative, and respectful throughout — as though \
   you are a knowledgeable analyst walking the patient through their test \
   results with patience and clarity. Do not be alarming, but do not soften \
   findings to the point of obscuring their significance.

8. Every recommendation or warning in the advisory paragraph must name the \
   specific test value it addresses and explain directly why that value makes \
   this recommendation or warning necessary. Generic advice with no traceable \
   link to a specific finding is not acceptable.
"""


USER_T2 = """The following result was produced by a heart disease classification \
model trained on cardiovascular test data. Using the structure defined in your \
instructions, analyse each test value and deliver the final advisory.

Prediction     : {prediction}

Cardiovascular test results:
  Age                    : {Age} years
  Sex                    : {Sex}
  Chest Pain Type        : {ChestPainType}
  Resting Blood Pressure : {RestingBP} mmHg
  Cholesterol            : {Cholesterol} mg/dL
  Fasting Blood Sugar    : {FastingBS} mg/dL
  Resting ECG            : {RestingECG}
  Max Heart Rate         : {MaxHR} bpm
  Exercise Angina        : {ExerciseAngina}
  Oldpeak                : {Oldpeak}
  ST Slope               : {STSlope}"""

def call_T2(inputs):
    inputs["prediction"] = get_prediction(inputs)
    
    response = call_gemini(
        SYSTEM_T2,
        USER_T2.format(**inputs)
    )
    
    from IPython.display import Markdown, display
    import re
    parts = response.split("---ADVISORY---")
    step1_tables   = parts[0].strip()
    tables = re.split(r'(?m)^(?=\|.*Test Value)', step1_tables)
    tables = [t.strip() for t in tables if t.strip()]
    step2_advisory = parts[1].strip() if len(parts) > 1 else ""

    display(Markdown(step2_advisory))
    display(Markdown("\nThe following represents more details regarding each of your cardiovascular test values:\n"))

    for table in tables:
       display(Markdown(table))

### Input/Output Examples

In [185]:
# Test Case 1: Heart Disease Detected 
inputs = {
    "prediction":       "",
    "Age":              55,
    "Sex":              "Male",
    "ChestPainType":  "ATA",
    "RestingBP":       140,
    "Cholesterol":      240,
    "FastingBS":       1,
    "RestingECG":      "Normal",
    "MaxHR":           150,
    "ExerciseAngina":  "Yes",
    "Oldpeak":          2.3,
    "STSlope":         "Flat"
}

call_T2(inputs)

Based on your cardiovascular test results, the findings indicate the presence of heart disease. Specifically, your Resting Blood Pressure of 140 mmHg is elevated to Stage 1 Hypertension and your Cholesterol level of 240 mg/dL is high, both of which are significant risk factors that require careful management through lifestyle adjustments and potentially medication. Furthermore, experiencing Exercise Angina, combined with an Oldpeak value of 2.3 and a flat ST Slope during your ECG, are strong indicators of reduced blood flow to your heart muscle during physical activity. These findings highlight the importance of discussing these results with your healthcare provider without delay to develop a comprehensive plan for further evaluation and treatment.


The following represents more details regarding each of your cardiovascular test values:


| Test Value          | Age: 55 years |
| :------------------ | :------------------------------------------------------------------------------------------------ |
| Assessment          | At 55 years old, you are considered to be in middle age, a period where cardiovascular health becomes an increasingly important focus. |

| Test Value          | Sex: Male |
| :------------------ | :------------------------------------------------------------------------------------------------ |
| Assessment          | You are male. Biological sex is a factor in cardiovascular risk assessment, with males generally having a higher risk at younger ages compared to females. |

| Test Value          | Chest Pain Type: ATA |
| :------------------ | :------------------------------------------------------------------------------------------------ |
| What It Measures    | This describes the nature of any discomfort or pain felt in your chest. |
| Assessment          | Your chest pain is classified as Atypical Angina (ATA), which refers to chest pain that may not present with all the typical characteristics of heart-related pain. |
| What It May Suggest | While "atypical" suggests it might not be classic heart-related pain, any chest pain still warrants careful consideration as part of the overall cardiovascular risk assessment. |

...

| Test Value          | Resting ECG: Normal |
| :------------------ | :------------------------------------------------------------------------------------------------ |
| What It Measures    | An electrocardiogram (ECG) records the electrical activity of your heart while you are at rest. |
| Assessment          | Your resting ECG results are described as normal. |
| What It May Suggest | A normal resting ECG generally indicates no significant electrical abnormalities in the heart at the time of the test, though it doesn't rule out all heart conditions. |

| Test Value          | Max Heart Rate: 150 bpm |
| :------------------ | :------------------------------------------------------------------------------------------------ |
| What It Measures    | This is the highest heart rate achieved during your exercise test. |
| Assessment          | A maximum heart rate of 150 beats per minute (bpm) was achieved during exertion. |
| What It May Suggest | This value is a reasonable response to exercise for your age, indicating your heart can increase its rate during physical exertion. |

| Test Value          | Exercise Angina: Yes |
| :------------------ | :------------------------------------------------------------------------------------------------ |
| What It Measures    | This indicates whether you experienced chest pain or discomfort specifically during physical exertion. |
| Assessment          | You reported experiencing angina, or chest pain, during exercise. |
| What It May Suggest | Angina that occurs with exercise is a significant symptom, often indicating that the heart muscle is not receiving enough blood flow during increased activity, which can be a sign of coronary artery disease. |

| Test Value          | Oldpeak: 2.3 |
| :------------------ | :------------------------------------------------------------------------------------------------ |
| What It Measures    | Oldpeak refers to the ST depression induced by exercise relative to the resting state, a measurement taken from an exercise electrocardiogram. |
| Assessment          | Your Oldpeak value is 2.3, which suggests a notable depression of the ST segment during physical exertion. |
| What It May Suggest | A significant Oldpeak value like this is a strong indicator of myocardial ischemia, meaning reduced blood flow to the heart muscle, especially during stress or exercise. Higher values typically suggest more severe ischemia. |

| Test Value          | ST Slope: Flat |
| :------------------ | :------------------------------------------------------------------------------------------------ |
| What It Measures    | This describes the pattern of the ST segment of the ECG during exercise, which helps assess how the heart responds to and recovers from exertion. |
| Assessment          | The slope of your ST segment during exercise is described as "flat." |
| What It May Suggest | A flat ST segment slope during exercise is considered an abnormal response, often associated with myocardial ischemia and can be an indicator of coronary artery disease. A healthy response would typically show an upsloping ST segment. |

In [187]:
# Test Case 2: No Heart Disease Detected - All Healthy Values
inputs = {
    "prediction":       "",
    "Age":              35,
    "Sex":              "Female",
    "ChestPainType":  "NAP",
    "RestingBP":       110,
    "Cholesterol":      180,
    "FastingBS":       0,
    "RestingECG":      "Normal",
    "MaxHR":           175,
    "ExerciseAngina":  "No",
    "Oldpeak":          0.0,
    "STSlope":         "Up"
}

call_T2(inputs)

Based on your cardiovascular test results, your overall heart health appears to be in a very good state, with no indications of heart disease. Your Resting Blood Pressure of 110 mmHg and Cholesterol level of 180 mg/dL are both optimal and desirable, respectively, reflecting excellent cardiovascular management. However, your Fasting Blood Sugar reading of 0 mg/dL is biologically implausible and prevents a proper assessment of this important metabolic indicator; it is crucial to have this test re-done to ensure all aspects of your cardiovascular risk are thoroughly evaluated. Continuing your current healthy lifestyle, which likely contributes to these positive results, is highly recommended to maintain your optimal heart health moving forward.


The following represents more details regarding each of your cardiovascular test values:


| Test Value | Age: 35 years |
| :------------------ | :-------------------------------------------------------------- |
| Assessment          | You are relatively young, which is a positive factor for cardiovascular health. |

| Test Value | Sex: Female |
| :------------------ | :-------------------------------------------------------------- |
| Assessment          | Biological sex can influence cardiovascular risk factors and presentation. |

| Test Value | Chest Pain Type: NAP |
| :------------------ | :-------------------------------------------------------------- |
| What It Measures    | This indicates the nature or type of chest discomfort experienced. |
| Assessment          | Your chest pain is classified as Non-Anginal Pain, suggesting it is unlikely to be directly linked to heart-related blockages. |
| What It May Suggest | This finding often indicates that if chest pain is present, it is less likely to be due to coronary artery disease. |

...

| Test Value | Resting ECG: Normal |
| :------------------ | :-------------------------------------------------------------- |
| What It Measures    | An electrocardiogram (ECG) records the electrical signals from your heart to check for various heart conditions. |
| Assessment          | Your resting ECG shows a normal pattern. |
| What It May Suggest | A normal ECG generally indicates that the heart's electrical activity is functioning as expected while at rest. |

| Test Value | Max Heart Rate: 175 bpm |
| :------------------ | :-------------------------------------------------------------- |
| What It Measures    | This is the highest heart rate achieved during maximal physical exertion. |
| Assessment          | At 35 years old, a maximum heart rate of 175 bpm is within an expected healthy range for someone exercising. |
| What It May Suggest | This indicates a healthy cardiac response and good capacity during physical activity. |

| Test Value | Exercise Angina: No |
| :------------------ | :-------------------------------------------------------------- |
| What It Measures    | This indicates whether you experience chest pain or discomfort specifically brought on by physical exertion. |
| Assessment          | You did not experience angina during exercise. |
| What It May Suggest | The absence of exercise-induced angina is a positive sign, suggesting sufficient blood flow to your heart muscle during increased demand. |

| Test Value | Oldpeak: 0.0 |
| :------------------ | :-------------------------------------------------------------- |
| What It Measures    | This value represents the ST segment depression induced by exercise, used to assess myocardial ischemia (reduced blood flow to the heart). |
| Assessment          | An Oldpeak value of 0.0 indicates no ST depression during exercise. |
| What It May Suggest | This is a very positive finding, suggesting good blood supply to the heart muscle even under stress. |

| Test Value | ST Slope: Up |
| :------------------ | :-------------------------------------------------------------- |
| What It Measures    | This describes the slope of the ST segment of the ECG during exercise, which provides information about heart muscle recovery. |
| Assessment          | Your ST segment shows an "Up" slope during exercise. |
| What It May Suggest | An "Up" slope is typically a normal and reassuring response to exercise, not suggesting significant blood flow issues. |

In [188]:
# Test Case 3: No Heart Disease Detected - Borderline Values
inputs = {
    "prediction":       "",
    "Age":              45,
    "Sex":              "Male",
    "ChestPainType":  "NAP",
    "RestingBP":       128,
    "Cholesterol":      210,
    "FastingBS":       0,
    "RestingECG":      "Normal",
    "MaxHR":           160,
    "ExerciseAngina":  "No",
    "Oldpeak":          0.8,
    "STSlope":         "Up"
}

call_T2(inputs)

Your cardiovascular assessment indicates no presence of heart disease. However, your resting blood pressure is 128 mmHg, categorized as "Elevated," meaning it's higher than optimal and should be monitored through regular checks and managed with a heart-healthy diet and increased physical activity to prevent it from progressing to high blood pressure. Similarly, your total cholesterol level of 210 mg/dL falls into the "borderline high" range; adopting a diet rich in fruits, vegetables, and whole grains while limiting saturated and trans fats can help bring this value into a healthier zone. Additionally, your Oldpeak value of 0.8, while mild, suggests a slight alteration in your heart's electrical activity during stress, further reinforcing the importance of maintaining a healthy lifestyle to support overall cardiovascular well-being. By proactively addressing these areas, you can reinforce your healthy status and continue to reduce your long-term cardiovascular risk.


The following represents more details regarding each of your cardiovascular test values:


| Test Value | 45 years |
|---|---|
| Assessment | Your age is 45 years. |

| Test Value | Male |
|---|---|
| Assessment | You are identified as male. |

| Test Value | Chest Pain Type: NAP |
|---|---|
| What It Measures | This indicates the type of chest pain experienced. |
| Assessment | Your chest pain type is Non-Anginal Pain (NAP), which is typically not related to heart issues. |
| What It May Suggest | This finding generally suggests that chest discomfort is less likely to be caused by a lack of blood flow to the heart. |

...

| Test Value | Resting ECG: Normal |
|---|---|
| What It Measures | An electrocardiogram (ECG) records the electrical activity of your heart at rest. |
| Assessment | Your resting ECG is normal. |
| What It May Suggest | A normal resting ECG indicates that your heart's electrical patterns are typical, showing no immediate signs of strain or damage. |

| Test Value | Max Heart Rate: 160 bpm |
|---|---|
| What It Measures | This is the fastest rate your heart reached during exercise. |
| Assessment | Your maximum heart rate during the test was 160 beats per minute (bpm). |
| What It May Suggest | This indicates a healthy heart rate response to physical activity for your age, suggesting good cardiovascular fitness. |

| Test Value | Exercise Angina: No |
|---|---|
| What It Measures | This indicates whether you experienced chest pain or discomfort during physical exertion. |
| Assessment | You did not experience exercise-induced angina, which is chest pain caused by reduced blood flow to the heart during activity. |
| What It May Suggest | The absence of exercise angina is a positive indicator, suggesting that your heart is receiving adequate blood flow during physical stress. |

| Test Value | Oldpeak: 0.8 |
|---|---|
| What It Measures | This value quantifies the amount of ST segment depression during exercise, which can indicate reduced blood flow to the heart. |
| Assessment | Your Oldpeak value is 0.8, which is a mild ST depression. |
| What It May Suggest | While mild, this finding suggests a slight alteration in your heart's electrical activity during stress, which is often considered in the context of other risk factors. |

| Test Value | ST Slope: Up |
|---|---|
| What It Measures | This describes the direction of the ST segment during exercise on an ECG, which can indicate heart health. |
| Assessment | Your ST slope is described as "Up," meaning it slopes upwards during exercise. |
| What It May Suggest | An upsloping ST segment is generally considered a normal or less concerning finding during exercise stress tests compared to other slopes. |

# Few-Shot Prompting

### Template ID
3

---

### Approach Definition

Few-shot prompting is a prompt engineering technique in which a language model is provided with a small number of labeled examples before being asked to perform a new but similar task. Instead of relying only on a direct instruction, the model learns the expected pattern of input-output behavior from the examples included in the prompt. These examples act as demonstrations that guide the model toward producing outputs in the required format and according to the intended logic (Brown et al., 2020).

---

### Relation to the Medical Field

Few-shot prompting is particularly well suited to the medical domain because clinical reasoning rarely happens in isolation. When a clinician assesses a new cardiovascular patient, they do not interpret the case from scratch. Instead, they draw on previously encountered patients with similar profiles, comparing the current case against familiar patterns before reaching a conclusion. A cardiologist who has seen many patients with high blood pressure, elevated cholesterol, and chest pain will immediately recognize that combination as a risk pattern worth taking seriously, not because of a single number, but because of how the full picture matches cases seen before(Elstein, 2002).

This is exactly how few-shot prompting works(Brown et al., 2020). By placing a small number of labeled patient records inside the prompt, the model is shown how similar cardiovascular profiles have been interpreted in the past. Each example acts as a reference point, guiding the model to recognize patterns across multiple variables rather than reacting to any single measurement in isolation. In that sense, few-shot prompting functions as a lightweight case-based reasoning mechanism, one that mirrors the comparative, pattern-driven logic that underlies real clinical judgment in cardiovascular care.

> Brown, T. et al. (2020). *Language Models are Few-Shot Learners*. NeurIPS.  
> https://arxiv.org/abs/2005.14165

> Elstein, A. S. (2002). *Clinical problem solving and diagnostic decision making*.  
> https://pmc.ncbi.nlm.nih.gov/articles/PMC1122649/

---

### Intended Use Case

**1- Deployment Environment**

The system is designed to be deployed as a heart disease advisory tool within a healthcare mobile application. Patients can access it through their smartphones after obtaining their clinical data, either from a hospital test or by manually entering their values into the system. The advisory is generated instantly and displayed on the same screen.



**2- The User**

The primary user of this template is a returning patient who has used the healthcare application before and has already received at least one prior advisory through the system. This user is familiar with the general style, tone, and structure of the generated response, and they return expecting a consistent and recognisable experience.
Unlike a first-time user, this patient does not need the system to introduce the format from the beginning. Instead, they benefit from receiving advice in a stable and predictable structure that allows them to compare their current result with previous advisories, notice changes in their condition, and follow their health status over time.

***Medical background:***  

No medical expertise is assumed. However, because the user has prior experience with the system, the response does not need to introduce or justify its structure, and can instead focus directly on the advisory content. The output should avoid complex terminology and focus on simple explanations supported by examples.

***Emotional state:***  
The patient may still feel some concern when reviewing a health-related result, but they are likely to approach the output with greater familiarity than a new user. For that reason, the response should remain supportive and clear while focusing on reassurance through consistency and readability.



**3- Inputs**

3.1 The patient heart disease classification result  

3.2 The patient clinical feature values  




**4- Output**

A structured but concise response that follows the pattern demonstrated in the examples. The output includes:
- A short interpretation of the patient’s condition  
- A brief explanation based on the most relevant features  
- A clear, simple tips and suggestions   

The response should remain consistent with the format shown in the examples, ensuring readability and ease of understanding.

---

### Design Rationale
**Advantage 1 —  Better adaptation to the task:**
Few-shot prompting helps the model adapt to the specific prediction task by showing it exactly how inputs and outputs should look. This is important in the heart disease dataset because the model needs to deal with structured patient attributes such as age, sex, chest pain type, resting blood pressure, cholesterol, fasting blood sugar, and other clinical indicators. A few examples help the model understand how these features are mapped to the target class.(OpenAI, 2023)

**Advantage 2 — Improved consistency of outputs:** 
When examples are included in the prompt, the model is more likely to respond in a consistent format. This is useful in a medical application because predictions should be presented clearly and uniformly. For example, if each example follows the same structure of patient data, prediction, and short explanation, the model is more likely to generate outputs that are easier to compare, evaluate, and interpret.

**Advantage 3 — Useful when labeled examples are meaningful:** 
Our dataset already contains labeled cases, which makes few-shot prompting a natural choice. Since the target variable indicates the presence or absence of heart disease, example records can be selected directly from the dataset to guide the model. This allows the prompting technique to align closely with the supervised learning setting, where the model benefits from seeing representative examples before handling unseen cases.



> OpenAI. (2023). *Prompt Engineering Guide*.  
> https://platform.openai.com/docs/guides/prompt-engineering



### Input / Output Example

#### System and User Prompt

In [13]:


SYSTEM_T3 = """
You are a cardiovascular advisory assistant supporting a heart disease prediction system.

Your task is to explain cardiovascular risk results to patients in a clear, concise, and supportive manner.
Follow the response style demonstrated in the examples.

Rules:
- Use simple patient-friendly language.
- Do not use technical jargon unless necessary.
- Do not mention probabilities, model confidence, or AI limitations.
- Do not provide a diagnosis.
- Do not invent clinical facts beyond the provided prediction and features.
- Keep the response to one paragraph.
- The response should include:
  1) a brief interpretation of the result,
  2) a short explanation based on the most relevant clinical features,
  3) a practical and reassuring advisory.
"""

USER_T3 = """
Below are example patient cases and their corresponding advisory responses.
Use them as a guide for wording, structure, tone, and level of detail.

Example 1
Input:
Prediction: No Heart Disease
Age: 42
Sex: Female
ChestPainType: NAP
RestingBP: 118
Cholesterol: 205
FastingBS: 0
RestingECG: Normal
MaxHR: 170
ExerciseAngina: No
Oldpeak: 0.3
STSlope: Up

Output:
The result suggests that there is no current strong indication of heart disease. Your overall clinical profile appears reassuring, particularly with a normal ECG, no exercise-related chest discomfort, and a strong heart rate response during activity. These factors are commonly associated with a lower cardiovascular risk profile. However, it is still important to maintain healthy habits such as regular exercise, balanced nutrition, and routine medical checkups to support long-term heart health.

Example 2
Input:
Prediction: Heart Disease
Age: 63
Sex: Male
ChestPainType: ASY
RestingBP: 155
Cholesterol: 270
FastingBS: 1
RestingECG: ST
MaxHR: 115
ExerciseAngina: Yes
Oldpeak: 3.0
STSlope: Flat

Output:
The result suggests an increased likelihood of heart disease, indicating that your cardiovascular health may require closer medical attention. Several factors may have contributed to this result, including elevated blood pressure, high cholesterol levels, exercise-related chest discomfort, and signs of stress on the heart during activity. These patterns are often associated with higher cardiovascular risk. It is strongly recommended that you consult a healthcare professional for further evaluation and follow appropriate guidance on lifestyle changes and risk management.

Example 3
Input:
Prediction: No Heart Disease
Age: 54
Sex: Male
ChestPainType: ATA
RestingBP: 138
Cholesterol: 245
FastingBS: 1
RestingECG: Normal
MaxHR: 148
ExerciseAngina: No
Oldpeak: 1.8
STSlope: Flat

Output:
The result indicates no current strong sign of heart disease, but some of your clinical values suggest that caution may still be needed. While there is no exercise-related angina and your ECG appears normal, other factors such as moderately elevated cholesterol, increased blood pressure, and higher stress-related indicators may still influence your overall cardiovascular risk. It would be beneficial to monitor your condition regularly and adopt healthy lifestyle practices to reduce potential future risk.

Now write a response for the following patient case in the same style.

Input:
Prediction: {prediction}
Age: {Age}
Sex: {Sex}
ChestPainType: {ChestPainType}
RestingBP: {RestingBP}
Cholesterol: {Cholesterol}
FastingBS: {FastingBS}
RestingECG: {RestingECG}
MaxHR: {MaxHR}
ExerciseAngina: {ExerciseAngina}
Oldpeak: {Oldpeak}
STSlope: {STSlope}

Output:
"""

from IPython.display import display, Markdown

def run_test_case(case_name, inputs):
    print(case_name)
    result = call_gemini(SYSTEM_T3, USER_T3.format(**inputs))
    display(Markdown(result))
    

#### Test case 1

In [18]:
# Test Case 1: Heart Disease Detected
inputs_1 = {
    "Age": 55,
    "Sex": "Male",
    "ChestPainType": "ATA",
    "RestingBP": 140,
    "Cholesterol": 240,
    "FastingBS": 1,
    "RestingECG": "Normal",
    "MaxHR": 150,
    "ExerciseAngina": "Yes",
    "Oldpeak": 2.3,
    "STSlope": "Flat"
}
inputs_1["prediction"] = get_prediction(inputs_1)
run_test_case("Test Case 1 — Heart Disease", inputs_1)


Test Case 1 — Heart Disease


The result suggests an increased likelihood of heart disease, indicating that your cardiovascular health may require closer medical attention. Several factors may have contributed to this result, including elevated blood pressure, high cholesterol levels, exercise-related chest discomfort, and signs of stress on the heart during activity, even with an otherwise normal resting ECG. These patterns are often associated with higher cardiovascular risk. It is strongly recommended that you consult a healthcare professional for further evaluation and follow appropriate guidance on lifestyle changes and risk management.

#### Test case 2

In [19]:
# Test Case 2: No Heart Disease Detected — All Healthy Values
inputs_2 = {
    "Age":              35,
    "Sex":              "Female",
    "ChestPainType":    "NAP",
    "RestingBP":        110,
    "Cholesterol":      180,
    "FastingBS":        0,
    "RestingECG":       "Normal",
    "MaxHR":            175,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.0,
    "STSlope":          "Up"
}
inputs_2["prediction"] = get_prediction(inputs_2)
run_test_case("Test Case 2 — No Heart Disease", inputs_2)


Test Case 2 — No Heart Disease


The result suggests that there is no current strong indication of heart disease, which is very reassuring. Your clinical profile appears excellent, particularly with healthy blood pressure and cholesterol levels, no exercise-related chest discomfort, a strong heart rate response during activity, and normal ECG readings and stress indicators. These factors are strongly associated with a healthy cardiovascular system. To maintain this positive outlook, it's important to continue with your healthy habits, including regular physical activity, balanced nutrition, and routine medical checkups for long-term well-being.

#### Test case 3

In [20]:

# Test Case 3: No Heart Disease Detected — Borderline Values
inputs_3 = {
    "Age":              45,
    "Sex":              "Male",
    "ChestPainType":    "NAP",
    "RestingBP":        128,
    "Cholesterol":      210,
    "FastingBS":        0,
    "RestingECG":       "Normal",
    "MaxHR":            160,
    "ExerciseAngina":   "No",
    "Oldpeak":          0.8,
    "STSlope":          "Up"
}
inputs_3["prediction"] = get_prediction(inputs_3)

run_test_case("Test Case 3 — No Heart Disease with Borderline Values", inputs_3)

Test Case 3 — No Heart Disease with Borderline Values


The result suggests that there is no current strong indication of heart disease. Your overall clinical profile appears reassuring, particularly with a normal ECG, no chest discomfort during exercise, and a healthy heart rate response during activity. These factors are commonly associated with a lower cardiovascular risk profile. However, it is still important to maintain healthy habits such as regular exercise, balanced nutrition, and routine medical checkups to support long-term heart health.